# Sparse Walker + validated SASRec/HSTU long-context results

Loads the already-saved SASRec/HSTU A100 table from Drive, then benchmarks the exact `FastWalkerRetriever` kernels from the current Sparse Walker benchmark on the same A100 runtime. No duplicated HSTU code, no gzip/base64 launcher.

In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

from pathlib import Path
import pandas as pd, numpy as np, torch

ROOT = Path("/content/drive/MyDrive/hstu_pure_pytorch_ml1m")
BASE = ROOT / "hstu_vs_sasrec_a100_long_context.csv"
assert BASE.exists(), f"Missing previous successful result: {BASE}"
base = pd.read_csv(BASE)
display(base)
print("GPU:", torch.cuda.get_device_name(0))


## Import the existing Sparse Walker benchmark implementation

In [ ]:
!git clone -q https://github.com/hanialshater/Sparsewalker-.git /content/Sparsewalker || true

import sys, importlib
sys.path.insert(0, "/content/Sparsewalker")
from pathlib import Path

# Import by file path so the benchmark script does not run its __main__ training loop.
script = Path("/content/Sparsewalker/sparsewalker_paper_benchmark_v6_hstu_fullce.py")
if not script.exists():
    script = Path("/content/Sparsewalker/experiments/sparsewalker_paper_benchmark_v6_hstu_fullce.py")
assert script.exists(), script

spec = importlib.util.spec_from_file_location("swbench", script)
sw = importlib.util.module_from_spec(spec)
sys.modules["swbench"] = sw
spec.loader.exec_module(sw)

# Match the FP32 streaming protocol used for the SASRec/HSTU table.
sw.AMP_DTYPE = torch.float32
sw.CFG.latency_warmup = 30
sw.CFG.latency_repeats = 200
print("Loaded:", script)
print("Walker geometry:", sw.CFG.d_model, sw.CFG.concept_side, sw.CFG.active_concepts, sw.CFG.graph_degree)


## Build random-weight Walker serving artifacts

Weights do not affect kernel workload. We use the exact model/artifact/kernel classes from the benchmark, but no training is required for latency.

In [ ]:
torch.manual_seed(42)
DEVICE = sw.DEVICE
N_ITEMS = 12_101
DEGREE = 128
model = sw.SparseWalkerModel(
    N_ITEMS, 50, d=64, layers=2, side=256, h=16,
    active=8, top_side=2, degree=4, fresh_weight=.25
).to(DEVICE).eval()

support = {"item": torch.randint(1, N_ITEMS+1, (model.n_concepts, DEGREE), dtype=torch.long)}
ret = sw.FastWalkerRetriever(model, support, DEGREE)
a = ret.a
item = torch.tensor([1], dtype=torch.int32, device=DEVICE)
sid = torch.randint(0, model.n_concepts, (8,), dtype=torch.int32, device=DEVICE)
sm = torch.full((8,), 1/8, dtype=torch.float32, device=DEVICE)

@torch.inference_mode()
def walker_state_step():
    sw.route_kernel[(1,)](item,a.item,a.query_weight,a.left_keys,a.right_keys,a.fresh_ids,a.fresh_mass,a.query,D=64,H=16,SIDE=256,INPUT_SCALE=8.,num_warps=4)
    sw.walk_kernel[(1,)](sid,sm,a.fresh_ids,a.fresh_mass,a.query,a.destination,a.edge,a.dest_key,a.next_ids,a.next_mass,K=8,DEGREE=4,H=16,num_warps=4)
    sw.readout_kernel[(1,)](item,a.next_ids,a.next_mass,a.concept_value,a.message_weight,a.norm_weight,a.norm_bias,a.item,a.hidden,K=8,D=64,INPUT_SCALE=8.,num_warps=4)
    return a.hidden

@torch.inference_mode()
def walker_terminal_step():
    return ret.step(item, sid, sm)

walker_state_step(); walker_terminal_step(); torch.cuda.synchronize()
print("Walker kernels compiled")


## Same CUDA-event timing protocol and merged table

In [ ]:
def bench(fn, warmup=30, iters=200):
    for _ in range(warmup): fn()
    torch.cuda.synchronize()
    vals=[]
    s=torch.cuda.Event(enable_timing=True); e=torch.cuda.Event(enable_timing=True)
    for _ in range(iters):
        s.record(); fn(); e.record(); e.synchronize(); vals.append(s.elapsed_time(e))
    x=np.asarray(vals)
    return dict(mean_ms=float(x.mean()), p50_ms=float(np.percentile(x,50)), p95_ms=float(np.percentile(x,95)), p99_ms=float(np.percentile(x,99)))

state_t = bench(walker_state_step)
term_t = bench(walker_terminal_step)
print("Walker state:", state_t)
print("Walker + d128 terminal:", term_t)

histories = sorted(base.history.unique())
wr=[]
for h in histories:
    wr.append({"model":"SparseWalker-state","backend":"exact-benchmark-Triton-FP32","history":h,"cache_MB":8*(4+4)/1024**2,**state_t})
    wr.append({"model":"SparseWalker-terminal-d128","backend":"exact-benchmark-Triton-FP32","history":h,"cache_MB":8*(4+4)/1024**2,**term_t})
walker_df=pd.DataFrame(wr)
combined=pd.concat([base,walker_df],ignore_index=True,sort=False)
display(combined)

p = combined.pivot_table(index="history", columns="model", values="p50_ms", aggfunc="first").reset_index()
if "SASRec" in p:
    p["Walker-state_speedup_vs_SASRec"] = p["SASRec"] / p["SparseWalker-state"]
    p["Walker-terminal_speedup_vs_SASRec"] = p["SASRec"] / p["SparseWalker-terminal-d128"]
if "HSTU-core" in p:
    p["Walker-state_speedup_vs_HSTU-core"] = p["HSTU-core"] / p["SparseWalker-state"]
    p["Walker-terminal_speedup_vs_HSTU-core"] = p["HSTU-core"] / p["SparseWalker-terminal-d128"]
display(p)

combined.to_csv(ROOT / "sasrec_hstu_walker_a100_streaming_same_harness.csv", index=False)
p.to_csv(ROOT / "sasrec_hstu_walker_a100_streaming_same_harness_speedups.csv", index=False)
print("Saved under:", ROOT)
